# Jeveration — few-shot frozen-feature RGBD

Uses the existing **95 training and 20 held-out Hypersim images**. Closed-form ridge/prototype initialization + 1,200 optimizer steps. No new data, no new model downloads except the GitHub source code. Checkpoints and report go to Drive. The full GPU training result is not yet verified.

In [ ]:
# @title 1. Mount Drive and install the repository
from google.colab import drive
drive.mount("/content/drive")
import sys, subprocess, torch
subprocess.run([sys.executable,"-m","pip","install","-q",
                "--upgrade","git+https://github.com/Apache0ne/Jeveration.git"],check=True)
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Use a BF16 L4 or A100"
print(torch.cuda.get_device_name(0))


In [ ]:
# @title 2. Verify exact existing dataset
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/Jeveration_GT/pilot")
CHECKPOINTS=ROOT/"checkpoints_frozen_probe"
ntrain=len(list((ROOT/"cache"/"train").glob("*.npz")))
nval=len(list((ROOT/"cache"/"val").glob("*.npz")))
print("Source:",ROOT,"Train:",ntrain,"Val:",nval)
assert ntrain==95 and nval==20, "The controlled comparison requires the same 95/20 dataset."
CHECKPOINTS.mkdir(parents=True,exist_ok=True)


In [ ]:
# @title 3. Analytic initialization + 1,200 independent updates (resumable)
import sys,subprocess
cmd=[sys.executable,"-u","-m","jeveration.rgbd_frozen_probe_train","fit",
     "--root",str(ROOT),"--save-dir",str(CHECKPOINTS),
     "--local-cache-dir","/content/jeveration_gt_pilot_cache",
     "--cache-mode","ram","--steps","1200","--batch-size","1",
     "--grad-accum","16","--projected","48","--init-stride","4",
     "--ridge-reg",".005","--lr-depth","0.0002",
     "--lr-normal","0.0003","--lr-seg","0.00025",
     "--eval-every","100","--save-every","200","--save-at","400,700,900"]
if (CHECKPOINTS/"latest_probe.json").exists():cmd.append("--resume")
print("Checkpoint destination:",CHECKPOINTS,flush=True)
subprocess.run(cmd,check=True)


In [ ]:
# @title 4. Export held-out geometry report ZIP
import sys,subprocess,shutil
from pathlib import Path
from google.colab import files
subprocess.run([sys.executable,"-u","-m","jeveration.rgbd_frozen_probe_train","report",
                "--root",str(ROOT),"--save-dir",str(CHECKPOINTS),
                "--local-cache-dir","/content/jeveration_gt_pilot_cache",
                "--output","/content/Jeveration_FrozenProbe_Report"],check=True)
source=Path("/content/Jeveration_RGBD_FrozenProbe_Report.zip")
assert source.is_file()
dest=ROOT/"reports"/source.name
dest.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(source,dest)
print("Saved:",dest)
files.download(str(source))
